# Graded Lab : Tool Use and Reflective Agents

In this lab, you will explore how AI agents can enhance research workflows by leveraging external tools and engaging in critical self-reflection. You'll learn how to build and integrate callable tools—such as web and academic search functions, and connect them to a language model using OpenAI's tool-calling API. Then, you’ll guide the agent to not only generate content but also **reflect** on its own output, improving the quality and depth of the final report. By the end of this lab, you will have implemented a mini agent capable of searching, reasoning, and publishing structured reports in HTML—laying the foundation for more advanced multi-step and autonomous AI systems.

### 🎯 Learning Objectives

By the end of this lab, you can:
- Chain steps into a research pipeline (**search → reflection → formatting**).
- Convert natural-language output into **styled HTML** suitable for sharing.

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to write your solution code or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the <span style="background-color: #0056D2; color: white; padding: 3px 5px; font-size: 16px; border-radius: 5px;">Submit assignment</span> button on the top right of the page.
---

In [2]:
# ================================
# Standard library imports
# ================================
import aisuite as ai
import json
import os

# ================================
# Third-party imports
# ================================
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, HTML

# ================================
# Local / project imports
# ================================
import research_tools

# ================================
# Environment setup
# ================================
load_dotenv()  # Load environment variables from .env file

# ================================
# Environment & Client
# ================================
from pathlib import Path

env_path = Path("../.env.local")
load_dotenv(env_path)

CLIENT = ai.Client(
    provider_configs={
        "openai": {
            "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
            "api_key": os.getenv("GEMINI_API_KEY"),
            
        }
    }
)

_PREFERED_MODEL = "openai:gemini-3.5-flash-lite"

In [3]:
import unittests

## Using Tools

You’ll use two research tools exposed in the `research_tools` module:
- **`arxiv_search_tool(query, max_results)`** – academic papers via arXiv API.
- **`tavily_search_tool(query, max_results, include_images)`** – general web search via Tavily.

Let's explore how the `arxiv_search_tool` works.

This tool searches arXiv and returns a list of papers with:
- `title`, `summary`, `url`, and (if available) `link_pdf`.

Below, we run a quick test and print the results in a readable format. Next cell is editable so feel free to try some search queries:


In [4]:
# Test the arXiv search tool
topic = "linear algebra"

arxiv_results = research_tools.arxiv_search_tool(topic, max_results=3)

# Show formatted arxiv_results
for i, paper in enumerate(arxiv_results, 1):
    if "error" in paper:
        print(f"❌ Error: {paper['error']}")
    else:
        print(f"📄 Paper {i}")
        print(f"  Title     : {paper['title']}")
        print(f"  URL       : {paper['url']}")
        print(f"  PDF       : {paper.get('link_pdf')}\n")


print("\n🧾 Raw arxiv_Results:\n")
print(json.dumps(arxiv_results, indent=2))

📄 Paper 1
  Title     : [2506.18666] Advanced linear algebra
  URL       : https://arxiv.org/abs/2506.18666
  PDF       : https://arxiv.org/pdf/2506.18666

📄 Paper 2
  Title     : [2508.21189] Faster Linear Algebra Algorithms with Structured Random Matrices
  URL       : https://arxiv.org/abs/2508.21189
  PDF       : https://arxiv.org/pdf/2508.21189

📄 Paper 3
  Title     : [2112.01898] Linear algebra with transformers
  URL       : https://arxiv.org/abs/2112.01898
  PDF       : https://arxiv.org/pdf/2112.01898


🧾 Raw arxiv_Results:

[
  {
    "title": "[2506.18666] Advanced linear algebra",
    "url": "https://arxiv.org/abs/2506.18666",
    "summary": "![archive](/static/base/1.0.1/images/arxiv-logo-primary-light.svg)\n\n# Mathematical Physics\n\n# Title:Advanced linear algebra\n\n|  |  |\n| --- | --- |\n| Comments: | 400 pages |\n| Subjects: | Mathematical Physics (math-ph); Functional Analysis (math.FA); History and Overview (math.HO); Operator Algebras (math.OA) |\n| Cite as: | [a

The `tavily_search_tool` calls the Tavily API to fetch web results. Returns a list of dicts:
- `title`, `content`, `url` (and optional image URLs when `include_images=True`).

Run the cell to inspect sample output. Next cell is editable so feel free to try some search queries:

In [5]:
# Test the Tavily search tool
topic = "retrieval-augmented generation applications"

tavily_results = research_tools.tavily_search_tool(topic)
for item in tavily_results:
    print(item)

{'title': 'Retrieval-augmented generation - Wikipedia', 'content': 'Finally, the LLM can generate output based on both the query and the retrieved documents. Some models incorporate extra steps to improve output, such as the re-ranking of retrieved information, context selection, and fine-tuning "Fine-tuning (deep learning)").\n\n## Applications\n\n[edit]\n\nRetrieval-augmented generation is used in applications where generated responses need to be grounded in external or frequently updated information.[citation needed] [...] Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. With RAG, LLMs first refer to a specified set of documents, then respond to user queries. These documents supplement information from the LLM\'s pre-existing training data. This allows LLMs to use domain-specific and/or updated information that is not available in the training data. For example, this 

## Tool Mapping

In the next cell you will define a dictionary that maps tool names (strings) to the actual Python functions. This allows the model to call tools by name during tool-calling. This dictionary will be used in your first graded function:

In [6]:
# Tool mapping
TOOL_MAPPING = {
    "tavily_search_tool": research_tools.tavily_search_tool,
    "arxiv_search_tool": research_tools.arxiv_search_tool,
}

## Exercise 1: Generate Research Report with Tools
**Goal:** Implement `generate_research_report_with_tools(prompt)`.
In this exercise, you'll work on a function that generates a detailed research report with the assistance of online tools. Focus on setting up interaction with the language model and handling the responses effectively.

## Key Hints

### 1. Setting Up the Chat with the Language Model
- **Tool Selection**: Ensure that the tools are automatically selected by the model. Look into how to set `tool_choice` to "auto" within the function call. A helpful resource can be found in [OpenAI’s Function Calling Documentation](https://platform.openai.com/docs/guides/function-calling#tool-choice).
- **Parameter Configuration**: Consider the parameters already defined in your function, such as model, messages, and tools. Think about how these might be used in your setup.

### 2. Recording Tool Call Results
- **Understanding the `ChatCompletionMessage`** object will help you access the required attributes to save the messages. An example of `ChatCompletionMessage` looks like this: 

```python
ChatCompletionMessage(
    content=None,
    refusal=None,
    role='assistant',
    annotations=[],
    audio=None,
    function_call=None,
    tool_calls=[
        ChatCompletionMessageFunctionToolCall(
            id='call_ymMki5TBB91efJhMPjgoqjop',
            function=Function(
                arguments='{"query":"radio observations of recurrent novae","max_results":5}',
                name='arxiv_search_tool'
            ),
            type='function'
        )
    ]
)
```
Assuming that `msg` if of type `ChatCompletionMessage`, if you wanted to get the `name` of a `tool_call` you can do something like:
```python
 for call in msg.tool_calls:
    tool_name = call.function.name
```
Finally, the `result` variable will be created by actually calling the function associated with each tool (`tool_func`).

By leveraging these hints, you'll work towards an implementation that enables robust data gathering and report generation through smart tool integration.

In [7]:
# GRADED FUNCTION: generate_research_report_with_tools
def generate_research_report_with_tools(prompt: str, model: str = _PREFERED_MODEL) -> str:
    """
    Generates a research report using OpenAI's tool-calling with arXiv and Tavily tools.

    Args:
        prompt (str): The user prompt.
        model (str): OpenAI model name.

    Returns:
        str: Final assistant research report text.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are a research assistant that can search the web and arXiv to write detailed, "
                "accurate, and properly sourced research reports.\n\n"
                "🔍 Use tools when appropriate (e.g., to find scientific papers or web content).\n"
                "📚 Cite sources whenever relevant. Do NOT omit citations for brevity.\n"
                "🌐 When possible, include full URLs (arXiv links, web sources, etc.).\n"
                "✍️ Use an academic tone, organize output into clearly labeled sections, and include "
                "inline citations or footnotes as needed.\n"
                "🚫 Do not include placeholder text such as '(citation needed)' or '(citations omitted)'."
            )
        },
        {"role": "user", "content": prompt}
    ]

    # List of available tools
    tools = [research_tools.arxiv_tool_def, research_tools.tavily_tool_def]

    # Maximum number of turns
    max_turns = 10
    
    # Iterate for max_turns iterations
    for _ in range(max_turns):

        ### START CODE HERE ###

        # Chat with the LLM via the client and set the correct arguments. Hint: Their names match names of variables already defined.
        # Make sure to let the LLM choose tools automatically. Hint: Look at the docs provided earlier!
        response = CLIENT.chat.completions.create( 
            model=model,
            messages=messages,
            tools=tools,
            tool_choice= "auto",
            temperature=1, 
        ) 

        ### END CODE HERE ###

        # Get the response from the LLM and append to messages
        msg = response.choices[0].message 
        messages.append(msg) 

        # Stop when the assistant returns a final answer (no tool calls)
        if not msg.tool_calls:      
            final_text = msg.content
            print("✅ Final answer:")
            print(final_text)
            break

        # Execute tool calls and append results
        for call in msg.tool_calls:
            tool_name = call.function.name
            args = json.loads(call.function.arguments)
            print(f"🛠️ {tool_name}({args})")

            try:
                tool_func = TOOL_MAPPING[tool_name]
                result = tool_func(**args)
            except Exception as e:
                result = {"error": str(e)}

            ### START CODE HERE ###

            # Keep track of tool use in a new message
            new_msg = { 
                # Set role to "tool" (plain string) to signal a tool was used
                "role": "tool",
                # As stated in the markdown when inspecting the ChatCompletionMessage object 
                # every call has an attribute called id
                "tool_call_id": call.id,
                # The name of the tool was already defined above, use that variable
                "name": tool_name,
                # Pass the result of calling the tool to json.dumps
                "content": json.dumps(result)
            }

            ### END CODE HERE ###

            # Append to messages
            messages.append(new_msg)

    return final_text

Run the following cell to check the correctness of your code. It might take a while so don't worry if it takes a couple of minutes to run:

In [8]:
# Test your code!
unittests.test_generate_research_report_with_tools(generate_research_report_with_tools)

🛠️ arxiv_search_tool({'query': 'radio observations of recurrent novae'})
✅ Final answer:
# Radio Observations of Recurrent Novae

## 1. Introduction and Astrophysical Context
Recurrent novae (RNs) are cataclysmic variable stars that experience periodic thermonuclear outbursts caused by thermonuclear runaways (TNR) in hydrogen-rich envelopes accreted onto a white dwarf (WD) from a binary companion [Mukai 2014, arXiv:1407.4526]. Unlike classical novae—which have only a single recorded eruption—recurrent novae feature exceptionally short recurrence times ranging from a decade to roughly a century. Theoretical modeling indicates that these short intervals require:
1. **A high-mass white dwarf** close to the Chandrasekhar limit (which has a smaller surface area and higher surface gravity, demanding a smaller critical envelope mass for a TNR), and
2. **A high accretion rate** driven either by a Roche-lobe-filling secondary or a dense stellar wind [Starrfield et al. 1985; Yaron et al. 2005].


## Exercise 2: Reflection + Rewrite

**Goal:** Implement `reflection_and_rewrite(report)`.

In this task, your goal is to develop a function that takes a report, analyzes it, generates a structured reflection, and produces an improved version of the report. This involves two main tasks: crafting a precise prompt and setting up a correctly configured response call to the language model.

## Key Steps

### 1. Create a User Prompt

- **Objective**: Guide the language model to output a structured response in JSON format.
- **Format**: Ensure the output includes two keys, `"reflection"` and `"revised_report"`.
- **Details**: Your reflection should cover strengths, limitations, suggestions, and opportunities. The revised report should incorporate these elements to improve clarity and academic tone.

### 2. Configure the Response Call

- **Parameters**: Use the specified model (e.g., `"gpt-4o-mini"`) and set the temperature equal to the `temperature` parameter of the graded function.
- **Structure**: Make sure the response setup directs the model properly, ensuring the JSON format is adhered to without additional commentary.


By implementing these steps, your function will effectively transform and improve the given reports. Handle JSON parsing carefully to ensure the output is valid and reliable. Happy coding!

In [66]:
# GRADED FUNCTION: reflection_and_rewrite
def reflection_and_rewrite(report, model: str = _PREFERED_MODEL, temperature: float = 0.3) -> dict:
    """
    
    Accepts raw text OR the messages list returned by generate_research_report_with_tools.

    Returns:
        dict with keys:
          - "reflection": structured reflection text
          - "revised_report": improved version of the input report
    """

    # Input can be plain text or a list of messages, this function detects and parses accordingly
    report = research_tools.parse_input(report)

    ### START CODE HERE ###

    # Define the prompt. A multi-line f-string is typically used for this.
    # Remember it should ask the model to output ONLY valid JSON with this structure:
    # {{ "reflection": "<text>", "revised_report": "<text>" }}
    user_prompt = f"""
You are an academic reviewer focused on analyzing scientific reports and proposing direct improvements.

Goal: To generate a structured reflection AND a revised, improved version of the provided report.

Strict Content Instructions:
1. In the "reflection" field, you MUST summarize using short bullet points: Strengths, Limitations, Suggestions, and Opportunities.
2. In the "revised_report" field, you must rewrite the report in an improved, but HIGHLY OBJECTIVE AND CONCISE manner to avoid truncation due to output token limits.

Output Format:
The output must be EXCLUSIVELY a valid JSON object, with no preamble, no explanations, and WITHOUT markdown code block formatting (such as ```json or ```). Start your response directly with the {{ character on the very first line and end with }} on the last line. Do not insert literal escape characters (like \\n) outside of the string values.

Strict Escape Rules for Valid JSON:
1. **LaTeX & Formulas:** If the text contains mathematical notation (LaTeX) that uses backslashes (e.g., \\approx, \\text, \\dot, \\gamma), you MUST double every single backslash (write \\\\approx, \\\\text, \\\\dot, \\\\gamma) inside the string values to maintain JSON compliance.
2. **Single Quotes:** NEVER escape single quotes with a backslash (do not write \\'). Write the single quote character directly (e.g., companion's).
3. **Double Quotes:** If you need to use double quotes inside the text fields, you MUST escape them using a backslash (\\\").

Desired JSON Structure:
{{
  "reflection": "Structured reflection text here.",
  "revised_report": "Synthesized and revised report text here."
}}

Report for analysis:
{report}
"""



    # Get a response from the LLM
    response = CLIENT.chat.completions.create( 
        # Pass in the model
        model=model,
        messages=[ 
            # System prompt is already defined
            {"role": "system", "content": "You are an academic reviewer and editor."},
            # Add user prompt
            {"role": "user", "content": user_prompt},
        ],
        # Set the temperature equal t
        temperature=temperature
    )

    ### END CODE HERE ###

    # Extract output
    llm_output = response.choices[0].message.content.strip()

    # Check if output is valid JSON
    try:
        data = json.loads(llm_output)
    except json.JSONDecodeError:
        #raise Exception("The output of the LLM was not valid JSON. Adjust your prompt.")
        print("The output of the LLM was not valid JSON. Adjust your prompt.")
        return response

    return {
        "reflection": str(data.get("reflection", "")).strip(),
        "revised_report": str(data.get("revised_report", "")).strip(),
    }

In [44]:
report = generate_research_report_with_tools("Explain the applications of retrieval-augmented generation in modern AI systems.")
reflection_result = reflection_and_rewrite(report)

🛠️ arxiv_search_tool({'query': 'Retrieval-Augmented Generation survey applications'})
✅ Final answer:
# Applications of Retrieval-Augmented Generation (RAG) in Modern AI Systems

**Retrieval-Augmented Generation (RAG)** has emerged as a cornerstone architecture in modern artificial intelligence [Lewis et al., 2020]. By combining the parametric memory of Large Language Models (LLMs) with the non-parametric memory of external knowledge bases (such as vector databases, enterprise search engines, and knowledge graphs), RAG overcomes core limitations of traditional LLMs—specifically, hallucination, static training horizons, and a lack of domain-specific context [Gao et al., 2023; Zhou et al., 2024].

This overview explores the primary applications of RAG across modern AI systems, categorized by domain and operational workflow.

---

## 1. Enterprise Knowledge Management and Intelligent Q&A

The most prominent adoption of RAG occurs within enterprise environments, where organizations need to

In [23]:
# Test your code!
unittests.test_reflection_and_rewrite(reflection_and_rewrite)

 All tests passed!


## Exercise 3: Convert Report to HTML
**Goal:** Implement `convert_report_to_html(report)`.
This exercise focuses on transforming a plain text research report into a well-structured HTML document. You will build a function to facilitate this conversion using a language model.

## Key Steps

### 1. Create a User Prompt
- **Objective**: Instruct the model to transform plain text into HTML structure.
- **Format**: Ensure the output is valid, clean HTML with appropriate section headers, formatted paragraphs, and clickable links.
- **Details**: Preserve the citation style and request that the model responds only with HTML, without additional commentary.

### 2. Configure the Response Call
- **Parameters**: Use the specified model (e.g., `"gpt-4o"`) and set an appropriate temperature to balance creativity and accuracy.
- **Structure**: Configure the `CLIENT.chat.completions.create` call properly, using both system and user prompts to ensure a clear and focused task description.

By following these steps, you'll effectively convert plaintext reports into formatted HTML documents.

In [46]:
# GRADED FUNCTION: convert_report_to_html
def convert_report_to_html(report, model: str = _PREFERED_MODEL, temperature: float = 0.5) -> str:
    """
    Converts a plaintext research report into a styled HTML page using OpenAI.
    Accepts raw text OR the messages list from the tool-calling step.
    """

    # Input can be plain text or a list of messages, this function detects and parses accordingly
    report = research_tools.parse_input(report)

    # System prompt is already provided
    system_prompt = "You convert plaintext reports into full clean HTML documents."

    ### START CODE HERE ###
    
    # Build the user prompt instructing the model to return ONLY valid HTML
    user_prompt = f"""
        You receive a report that can be a plain text or a list of messages to convert into full clean HTML documents
        report: {report}
        youmust return ONLY valid HTML.
    """

    # Call the LLM by interacting with the CLIENT. 
    # Remember to set the correct values for the model, messages (system and user prompts) and temperature
    response = CLIENT.chat.completions.create( 
        # Pass in the model
        model=model,
        messages=[ 
            # System prompt is already defined
            {"role": "system", "content": system_prompt},
            # Add user prompt
            {"role": "user", "content": user_prompt},
        ],
        # Set the temperature equal t
        temperature=temperature
    )

    ### END CODE HERE ###

    # Extract the HTML from the assistant message
    html = response.choices[0].message.content.strip()  

    return html

In [47]:
# Test your code!
unittests.test_convert_report_to_html(convert_report_to_html)

 All tests passed!


### 🚀 End-to-End Pipeline

Run this cell to execute the full workflow:

1. Generate a research report (tools).
2. Reflect on the report.
3. Convert the report to HTML.

> You should see the rendered HTML below and two concise reflections in the console.

In [67]:
# 1) Research with tools
prompt_ = "Radio observations of recurrent novae"
preliminary_report = generate_research_report_with_tools(prompt_)
print("=== Research Report (preliminary) ===\n")
print(preliminary_report)

# 2) Reflection on the report (use the final TEXT to avoid ambiguity)
reflection_text = reflection_and_rewrite(preliminary_report)   # <-- pass text, not messages
print("=== Reflection on Report ===\n")
print(reflection_text['reflection'], "\n")
print("=== Revised Report ===\n")
print(reflection_text['revised_report'], "\n")


# 3) Convert the report to HTML (use the TEXT and correct function name)
html = convert_report_to_html(reflection_text['revised_report'])

print("=== Generated HTML (preview) ===\n")
print((html or "")[:600], "\n... [truncated]\n")

# 4) Display full HTML
display(HTML(html))

🛠️ arxiv_search_tool({'query': 'radio observations of recurrent novae'})
🛠️ tavily_search_tool({'query': 'radio observations of recurrent novae review'})
✅ Final answer:
# Radio Observations of Recurrent Novae

Recurrent novae (RNe) are cataclysmic variable stars that experience periodic thermonuclear eruptions on the surface of an accreting white dwarf (WD) with recurrence timescales ranging from decades to a century. Unlike classical novae, which have only ever been observed to erupt once in recorded history, RNe typically feature massive WDs (often near the Chandrasekhar limit) accreting matter at high rates from a companion star. 

Radio observations provide a powerful, unobstructed window into these explosive events. While optical and UV emission can suffer heavily from internal and interstellar dust extinction, radio waves freely propagate through the circumstellar medium (CSM). Consequently, radio light curves and spectra are essential for mapping the geometry of nova ejecta, tr

### 📌 “Expected Output” note (for the notebook text cell)

- `generate_research_report_with_tools` should return a **non-trivial string** (> 50 chars).

- `reflection_and_rewrite` should return a **dict** with **'reflection'** and **'revised\_report'** (both strings). The reflection should **mention** the four sections (Strengths, Limitations, Suggestions, Opportunities).

- `convert_report_to_html` should return a **string that looks like HTML** (e.g., includes `<html>`, `<h1>`, `<p>`, or closing tags).


---

## ✅ Wrap-Up

You built a mini research agent that can:
- 🔎 call tools (arXiv + Tavily),
- 🧠 reflect on its own output,
- 📰 publish a clean HTML report.

Great job!

### What to Submit
- Your notebook with Exercise 1–3 completed.

### Troubleshooting (quick)
- **Model/tool-call loop stalls?** Lower `max_turns` or print intermediate messages.
- **HTML looks odd?** Re-run conversion with a fresh assistant response.

**You’re done—nice work!** 🚀
